In [47]:
import math
import os
import scipy
from scipy.optimize import lsq_linear
import numpy as np
from scipy.linalg import null_space
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, halfnorm
import random
from scipy.io import loadmat
import random
import pickle
import sys
from sklearn.linear_model import RidgeCV
sys.path.append(r"c:\Users\katie\OneDrive\Documents\GitHub\trial")
import PCA_Regress as pcar
from brokenaxes import brokenaxes
from matplotlib.gridspec import GridSpec

In [3]:
base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_neu.pkl', "rb") as input_file:
    J_pickle = pickle.load(input_file)
del input_file

file_path = os.path.join(base_path, 'N_neu.pkl')
with open(file_path, "rb") as input_file:
    N_pickle = pickle.load(input_file)
del input_file

base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_mus.pkl', "rb") as input_file:
    J_pickle_m = pickle.load(input_file)
del input_file

ile_path = os.path.join(base_path, 'N_mus.pkl')
with open(ile_path, "rb") as input_file:
    N_pickle_m = pickle.load(input_file)
del input_file

# base_path = "/Users/kb6113/Desktop/Thesis"
# with open(base_path+'/J_neu.pkl', "rb") as input_file:
#     J_pickle = pickle.load(input_file)
# del input_file

# with open(base_path+'/J_mus.pkl', "rb") as input_file:
#     J_pickle_m = pickle.load(input_file)
# del input_file

J_all_tensor = J_pickle['J_all']['interpPSTH']
J_M1_tensor = J_pickle['J_M1']['interpPSTH']
J_PMd_tensor = J_pickle['J_PMd']['interpPSTH']
J_idx = np.r_[0:18, 36:45]
J_ntm_tensor = J_all_tensor[J_idx, :, :]
J_mus_tensor = J_pickle_m['interpPSTH']


N_all_tensor = N_pickle['N_all']['interpPSTH']
N_M1_tensor = N_pickle['N_M1']['interpPSTH']
N_PMd_tensor = N_pickle['N_PMd']['interpPSTH']
N_mus_tensor = N_pickle_m['interpPSTH']

<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
C:\Users\katie\AppData\Local\Temp\ipykernel_107120\3800015342.py:2: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_neu.pkl', "rb") as input_file:
C:\Users\katie\AppData\Local\Temp\ipykernel_107120\3800015342.py:12: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_mus.pkl', "rb") as input_file:


In [45]:
def red_rank (tensor_N, tensor_M, rank): 
    """
    Running reduced rank regression 
    """

    # splicing preparatory and movement bins and scaling frmo 0 to 1 and mean centering
    Neu_pm, Neu_move, M_move = pcar.time_shift(tensor_N, tensor_M)

    # finding W_ls 
    cov = Neu_move.T @ Neu_move
    print(cov.shape)
    W_LS = np.linalg.solve(cov, Neu_move.T @ M_move)
    
    
    # running PCA on Neu_move @ W
    pre_PCA = W_LS.T @ Neu_move.T @ Neu_move @ W_LS 
    U, S_, V_T = np.linalg.svd(pre_PCA)
    V_rank = V_T.T[:,:rank]
    
    W_RRR = W_LS @ V_rank @ V_rank.T

    return W_RRR, V_rank, W_LS



In [52]:
rank = 3
W_RRR, V_rank, W_LS = red_rank(N_all_tensor, N_mus_tensor, rank)
U_RRR = W_LS @ V_rank
Q_potent, _ = np.linalg.qr(U_RRR)
Q_null = null_space(Q_potent.T)

hold = W_LS @ V_rank
U, S, V = np.linalg.svd(hold)
print(U[0, :rank])
print(Q_potent[0,:])
print(Q_null.shape)


(275, 275)
[ 0.04366438 -0.08151177 -0.28148414]
[-0.00600954  0.1126563  -0.27396444]
(275, 272)
